# Index Building & Hypothesis Testing

**Scope:** Build composite indices from topic groups and test hypotheses about relationships between topic distributions and book ratings across tiers.

**Key Objectives:**
1. **Index Building**: Construct composite indices from taxonomy groups and topics
2. **Hypothesis Testing**: Test specific hypotheses about topic-rating relationships
3. **Statistical Validation**: Apply appropriate statistical tests and corrections

**Statistical Procedures:**
- **Kruskal-Wallis test**: Non-parametric test for differences across 3+ groups (Top/Middle/Trash)
- **Mann-Whitney U test**: Pairwise comparisons between tiers
- **Cliff's Delta**: Effect size for ordinal data (interpretation: |δ| < 0.147 = negligible, 0.147-0.33 = small, 0.33-0.474 = medium, > 0.474 = large)
- **Epsilon-squared**: Effect size for Kruskal-Wallis (interpretation: ε² < 0.01 = negligible, 0.01-0.06 = small, 0.06-0.14 = medium, > 0.14 = large)
- **Holm correction**: Multiple comparison correction for pairwise tests

**Outputs:** `results/stage10_correlation_analysis/indexing_hypothesis_testing/` (figures, tables, indices)


In [1]:
# 1. Setup & Imports
# NOTE: Always use venv for Python commands
# If running from terminal: source venv/bin/activate

from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

# ----------------------------
# Global plotting style + color policy
# ----------------------------
import matplotlib as mpl
import matplotlib.ticker as mtick
from matplotlib.colors import TwoSlopeNorm

# Set plotting defaults
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})

# Inline plotting for notebook
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import warnings
warnings.filterwarnings('ignore', message='.*IProgress not found.*')
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm.auto')

PROJECT_ROOT = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")

# --- Tier policy (ALWAYS) ---
TIER_ORDER_RAW = ["top", "middle", "trash"]
TIER_ORDER_DISPLAY = ["Top", "Middle", "Trash"]
TIER_MAP = {"top": "Top", "middle": "Middle", "trash": "Trash"}

TIER_COLORS = {"top": "#d62728", "middle": "#ff7f0e", "trash": "#1f77b4"}  # Top red, Trash blue
TIER_COLORS_DISPLAY = {"Top": TIER_COLORS["top"], "Middle": TIER_COLORS["middle"], "Trash": TIER_COLORS["trash"]}

# --- Effect/heatmap policy (ALWAYS) ---
# 1) For "temperature" (magnitudes like shares): high = red
HEATMAP_MAG_CMAP = "Reds"   # sequential: higher -> darker red

# 2) For effects (Top - Trash): positive = red, negative = blue, centered at 0
EFFECT_CMAP_MPL = "RdBu_r"  # diverging with red on positive side
EFFECT_CMAP_PLOTLY = [(0.0, "#2166ac"), (0.5, "#f7f7f7"), (1.0, "#b2182b")]  # blue -> white -> red

def ensure_tier_display(df, raw_col="rating_tier", out_col="Tier"):
    df = df.copy()
    df[out_col] = df[raw_col].map(TIER_MAP)
    df[out_col] = pd.Categorical(df[out_col], categories=TIER_ORDER_DISPLAY, ordered=True)
    return df

def symmetric_norm(values, fallback=0.25):
    """
    Returns (norm, lim) where lim = max(|values|) and norm centers at 0.
    For diverging colormaps (effects, deltas).
    """
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    lim = fallback if v.size == 0 else float(np.max(np.abs(v)))
    lim = max(lim, 1e-9)
    return TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim), lim

def symmetric_effect_norm(values, default=0.35):
    """
    Returns (norm, lim) where lim = max(|values|) and norm centers at 0.
    Alias for symmetric_norm for backward compatibility.
    """
    return symmetric_norm(values, fallback=default)

print("✓ Global plotting style and color policy defined")


✓ PROJECT_ROOT: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor
✓ Global plotting style and color policy defined


## 2. Statistical Helper Functions

**Cliff's Delta**: Non-parametric effect size measure for ordinal data. Compares the proportion of times values in one group exceed values in another group.

**Epsilon-squared**: Effect size for Kruskal-Wallis test, analogous to eta-squared for ANOVA. Measures the proportion of variance explained by group membership.

In [2]:
def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """
    Cliff's delta effect size for ordinal data.
    
    Interpretation:
    - |δ| < 0.147: negligible
    - 0.147 ≤ |δ| < 0.33: small
    - 0.33 ≤ |δ| < 0.474: medium
    - |δ| ≥ 0.474: large
    """
    x = np.asarray(x)
    y = np.asarray(y)
    gt = np.sum(x[:, None] > y[None, :])
    lt = np.sum(x[:, None] < y[None, :])
    return (gt - lt) / (len(x) * len(y))


def compute_epsilon_squared(kruskal_stat: float, n: int, k: int) -> float:
    """
    Epsilon-squared effect size for Kruskal-Wallis test.
    
    Interpretation:
    - ε² < 0.01: negligible
    - 0.01 ≤ ε² < 0.06: small
    - 0.06 ≤ ε² < 0.14: medium
    - ε² ≥ 0.14: large
    """
    return (kruskal_stat - (k - 1)) / (n - k)


def unify_book_id_dtype(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure book_id is Int64 dtype."""
    if 'book_id' in df.columns:
        df['book_id'] = pd.to_numeric(df['book_id'], errors='coerce').astype('Int64')
    return df

print("✓ Helper functions defined")


✓ Helper functions defined


## 3. Paths & Output Configuration

Setup data input paths and output directories. Outputs will be saved to a dedicated subfolder for this analysis.


In [3]:
# Data preparation directories
DATA_PREP_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "data_preparation"
BOOK_FEATURES_DIR = DATA_PREP_DIR / "book_features"
TOPIC_PROBS_DIR = DATA_PREP_DIR / "topic_probabilities"
TAXONOMY_RADWAY_DIR = DATA_PREP_DIR / "taxonomy_radway_eda"

# Topic analysis directories (for input data)
TOPIC_ANALYSIS_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "topic_analysis_all_368"
TOPIC_TABLES_DIR = TOPIC_ANALYSIS_DIR / "tables"

# Taxonomy group analysis (for input data)
TAXONOMY_GROUP_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "taxonomy_group_analysis"
TAXONOMY_GROUP_TABLES_DIR = TAXONOMY_GROUP_DIR / "tables"

# Data file paths
TOPIC_LOOKUP_PATH = TAXONOMY_RADWAY_DIR / "topic_lookup.parquet"
AUTHOR_DOMINANCE_PATH = TOPIC_TABLES_DIR / "topic_author_dominance.parquet"
BOOK_TOPIC_PROBS_PATH = TOPIC_PROBS_DIR / "book_topic_probs.parquet"
TOPIC_HEALTH_PATH = TOPIC_TABLES_DIR / "topic_health_table.parquet"
BOOK_WIDE_PATH = BOOK_FEATURES_DIR / "book_taxonomy_main_props_wide.parquet"
GOODREADS_PATH = PROJECT_ROOT / "data" / "processed" / "goodreads.csv"
TOPIC_TAXONOMY_MAPPING_PATH = TAXONOMY_GROUP_TABLES_DIR / "topic_taxonomy_mapping.csv"

# Output directories (new dedicated subfolder)
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "indexing_hypothesis_testing"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
INDEX_DIR = OUTPUT_DIR / "indices"

for d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, INDEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Data paths:")
print(f"  Topic lookup: {TOPIC_LOOKUP_PATH}")
print(f"  Author dominance: {AUTHOR_DOMINANCE_PATH}")
print(f"  Book topic probs: {BOOK_TOPIC_PROBS_PATH}")
print(f"  Topic health: {TOPIC_HEALTH_PATH}")
print(f"  Topic taxonomy mapping: {TOPIC_TAXONOMY_MAPPING_PATH}")
print(f"\nOutput directory: {OUTPUT_DIR}")


Data paths:
  Topic lookup: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/taxonomy_radway_eda/topic_lookup.parquet
  Author dominance: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/topic_analysis_all_368/tables/topic_author_dominance.parquet
  Book topic probs: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/topic_probabilities/book_topic_probs.parquet
  Topic health: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/topic_analysis_all_368/tables/topic_health_table.parquet
  Topic taxonomy mapping: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/ta

## 4. Helper Functions: Parsing & Tag Maps

Helper functions for parsing category fields and building tag dimension maps.


In [4]:
import json
import re
import difflib

def _try_json(s: str):
    s = s.strip()
    if not s:
        return None
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        try:
            return json.loads(s)
        except Exception:
            return None
    return None

def parse_cat_field(x) -> set:
    """Parse primary_categories / secondary_categories into a set of tokens."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return set()
    if isinstance(x, (list, tuple, set)):
        return {str(t).strip() for t in x if str(t).strip()}
    s = str(x).strip()
    if not s:
        return set()
    js = _try_json(s)
    if isinstance(js, list):
        return {str(t).strip() for t in js if str(t).strip()}
    parts = re.split(r"[,\n;|]+", s)
    return {p.strip() for p in parts if p.strip()}

def secondary_dim_map(sec_set: set) -> dict:
    """
    Convert {'setting:bedroom','activity:kissing','sexual:touching'} ->
    {'setting':{'bedroom'}, 'activity':{'kissing'}, 'sexual':{'touching'}}
    """
    out = {}
    for tok in sec_set:
        if ":" in tok:
            dim, val = tok.split(":", 1)
            dim, val = dim.strip(), val.strip()
            if dim and val:
                out.setdefault(dim, set()).add(val)
        else:
            out.setdefault("_flat", set()).add(tok)
    return out

print("✓ Parsing helper functions defined")


✓ Parsing helper functions defined


## 5. Rule Engine: include_any, include_all, exclude_any

Extended rule engine supporting:
- `include_any`: OR across signals (backward compatible with plain dict)
- `include_all`: AND across signals (kept uncommented)
- `exclude_any`: OR exclusions


In [5]:
def has_any_in_set(s: set, needles: list) -> bool:
    """Check if set s contains any of the needles."""
    if not isinstance(s, set):
        return False
    return any(n in s for n in needles)


def str_contains_any(text: str, needles: list) -> bool:
    """Check if text contains any of the needles (case-insensitive substring)."""
    t = (text or "").lower()
    return any(n.lower() in t for n in needles)


def sec_has(dim_map: dict, dim: str, vals: list) -> bool:
    """Check if secondary_dim dict has dim:val for any val in vals."""
    got = dim_map.get(dim, set())
    return any(v in got for v in vals)


def _rule_to_mask(df: pd.DataFrame, rule: dict) -> pd.Series:
    """
    Build a boolean mask for a *single* rule dict using OR across the keys inside that dict.
    
    Keys supported:
      taxonomy_subgroups (uses tax_sub_name - 28 subgroups),
      taxonomy_groups (uses tax_main_name - 7/8 main groups),
      primary_any, secondary_any,
      secondary_dim_any={dim:[vals]},
      radway_phase_any (substring), radway_main_any (substring),
      keyword_any (substring over label/scene_summary/keywords)
    """
    parts = []
    
    # taxonomy "node" (your 28 subcategories, unified: tax_sub_name if present else tax_main_name)
    if "taxonomy_subgroups" in rule and "tax_node_name" in df.columns:
        parts.append(df["tax_node_name"].isin(rule["taxonomy_subgroups"]))
    
    # taxonomy main group (7/8)
    if "taxonomy_groups" in rule and "tax_group_name" in df.columns:
        parts.append(df["tax_group_name"].isin(rule["taxonomy_groups"]))
    
    if "primary_any" in rule:
        needles = rule["primary_any"]
        parts.append(df["primary_set"].apply(lambda s: has_any_in_set(s, needles)))
    
    if "secondary_any" in rule:
        needles = rule["secondary_any"]
        parts.append(df["secondary_set"].apply(lambda s: has_any_in_set(s, needles)))
    
    if "secondary_dim_any" in rule:
        dim_any = rule["secondary_dim_any"]
        parts.append(df["secondary_dim"].apply(
            lambda m: any(sec_has(m, dim, vals) for dim, vals in dim_any.items())
        ))
    
    if "radway_phase_any" in rule and "radway_phase_name" in df.columns:
        needles = rule["radway_phase_any"]
        parts.append(df["radway_phase_name"].fillna("").apply(lambda t: str_contains_any(t, needles)))
    
    if "radway_main_any" in rule and "radway_main_name" in df.columns:
        needles = rule["radway_main_any"]
        parts.append(df["radway_main_name"].fillna("").apply(lambda t: str_contains_any(t, needles)))
    
    if "keyword_any" in rule:
        needles = rule["keyword_any"]
        combo = (
            df.get("label", "").fillna("").astype(str) + " " +
            df.get("scene_summary", "").fillna("").astype(str) + " " +
            df.get("keywords", "").fillna("").astype(str)
        )
        parts.append(combo.apply(lambda t: str_contains_any(t, needles)))
    
    if not parts:
        return pd.Series(False, index=df.index)
    
    mask = parts[0].copy()
    for p in parts[1:]:
        mask |= p
    return mask


def build_mask_logic(df: pd.DataFrame, spec: dict) -> pd.Series:
    """
    Supports both formats:
    
    Old:
      {"include": {...}, "exclude": {...}}
    
    New:
      {"include_any": [... or dict], "include_all": [...], "exclude_any": [... or dict]}
    """
    
    # --- alias old keys to new keys ---
    if "include" in spec and "include_any" not in spec:
        spec = dict(spec)  # shallow copy
        spec["include_any"] = spec.pop("include")
    
    if "exclude" in spec and "exclude_any" not in spec:
        spec = dict(spec)
        spec["exclude_any"] = spec.pop("exclude")
    
    # Backward compatibility: if user passes a plain rule dict (no logic keys), treat it as include_any
    if not any(k in spec for k in ["include_any", "include_all", "exclude_any"]):
        spec = {"include_any": spec}
    
    # ----- include_any -----
    inc_any = spec.get("include_any", None)
    if inc_any is None:
        inc_mask = pd.Series(False, index=df.index)
    else:
        if isinstance(inc_any, dict):
            inc_any = [inc_any]
        inc_mask = pd.Series(False, index=df.index)
        for rule in inc_any:
            inc_mask |= _rule_to_mask(df, rule)
    
    # ----- include_all (AND across items; each item ORs its internal keys) -----
    inc_all = spec.get("include_all", [])
    if isinstance(inc_all, dict):
        inc_all = [inc_all]
    for rule in inc_all:
        inc_mask &= _rule_to_mask(df, rule)
    
    # ----- exclude_any -----
    exc_any = spec.get("exclude_any", None)
    if exc_any is not None:
        if isinstance(exc_any, dict):
            exc_any = [exc_any]
        exc_mask = pd.Series(False, index=df.index)
        for rule in exc_any:
            exc_mask |= _rule_to_mask(df, rule)
        inc_mask &= (~exc_mask)
    
    return inc_mask

print("✓ Rule engine defined")


✓ Rule engine defined


## 5.5. Taxonomy-First Composite Builder

Taxonomy-first mask builder that ensures composites are anchored to taxonomy subgroups, with tags used only for refinement.


In [6]:
def resolve_subgroups(requested: list, available: list) -> tuple:
    """
    Try to resolve requested subgroup names to what's actually in topic_spine.
    Returns (resolved, missing_suggestions).
    """
    avail_set = set(available)
    resolved = []
    missing = []
    
    lower_avail = {a.lower(): a for a in available}
    
    for name in requested:
        if name in avail_set:
            resolved.append(name)
            continue
        if name.lower() in lower_avail:
            resolved.append(lower_avail[name.lower()])
            continue
        
        # contains match
        contains = [a for a in available if name.lower() in a.lower()]
        if len(contains) == 1:
            resolved.append(contains[0])
            continue
        
        # fuzzy suggestions
        sugg = difflib.get_close_matches(name, available, n=5, cutoff=0.55)
        missing.append((name, sugg))
    
    return resolved, missing


def taxonomy_first_mask(
    df: pd.DataFrame,
    core_subgroups: list,
    refine_spec: dict = None,
    fallback_spec: dict = None,
    force_taxonomy_share: float = 0.0,
    mass_col: str = "prevalence",
    weight_col: str = "w_eff",
    name: str = ""
) -> tuple:
    """
    Taxonomy-first:
      base_core = topics in core_subgroups (from tax_sub_name - 28 subgroups)
      refined = base_core AND refine_spec (if provided)
      fallback = fallback_spec (tag-only catch, kept minimal and audited)
      final = refined OR fallback
    
    Also returns audit dict including "share_core_mass".
    
    force_taxonomy_share:
      if > 0, prints a warning when the final mask's mass from core is < threshold.
      (doesn't auto-fix; you tighten specs manually)
    """
    if "tax_node_name" not in df.columns:
        raise ValueError("tax_node_name missing — unified taxonomy spine not built. Run cell 27 (unified taxonomy node) first.")
    
    base_core = df["tax_node_name"].isin(core_subgroups)
    refined = base_core.copy()
    
    if refine_spec is not None:
        refined &= build_mask_logic(df, refine_spec)
    
    final = refined.copy()
    fb = pd.Series(False, index=df.index)
    
    if fallback_spec is not None:
        fb = build_mask_logic(df, fallback_spec)
        final |= fb
    
    # audit mass share
    mass = df[mass_col].fillna(0.0) if mass_col in df.columns else pd.Series(1.0, index=df.index)
    w = df[weight_col].fillna(1.0) if weight_col in df.columns else pd.Series(1.0, index=df.index)
    
    total_mass = float((mass * w * final.astype(float)).sum())
    core_mass = float((mass * w * (final & base_core).astype(float)).sum())
    share_core = core_mass / (total_mass + 1e-12)
    
    audit = {
        "composite": name,
        "n_topics_final": int(final.sum()),
        "n_topics_core_in_final": int((final & base_core).sum()),
        "n_topics_fallback_only": int((final & fb & (~base_core)).sum()),
        "total_mass": total_mass,
        "core_mass": core_mass,
        "share_core_mass": share_core,
    }
    
    if force_taxonomy_share > 0 and total_mass > 0 and share_core < force_taxonomy_share:
        print(f"⚠️  [{name}] core mass share {share_core:.2f} < {force_taxonomy_share:.2f}. Tighten fallback/refine rules.")
    
    return final, audit

print("✓ Taxonomy-first mask builder defined")


✓ Taxonomy-first mask builder defined


## 6.5. Taxonomy-First Composite Definitions

Define composites using taxonomy-first approach: taxonomy subgroups as core, tags for refinement.


In [7]:
# -----------------------------
# Tag lexicons (your real vocab)
# -----------------------------

EXPLICIT_SEXUAL_DIM = [
    "affair","clitoral_stimulation","compromising_positions","dominatrix","eyes_on_me",
    "hair_pulling","nudity","oral_sex","touching","waves_of_orgasm"
]
EXPLICIT_ACTIVITIES = [
    "oral_sex", "mutual_stimulation", "nipple_play", "BDSM",
    "checking_condom", "comparing_sizes"
]
EXPLICIT_KEYWORDS = [
    "nipple", "nipples", "clit", "clitoral", "orgasm", "condom", "thrust",
    "penetrat", "cum", "blowjob", "lick", "spank", "dominatrix"
]

# Soft erotic / attraction (B) includes foreplay by your decision
SOFT_AFFECTION_ACTIVITIES = [
    "kissing", "making_out", "foreplay", "flirting", "gazing", "eye_contact",
    "smiling", "laughing", "teasing", "knowing_glance", "whispering", "staring_and_smiling"
]

# Compatibility alias for old code
NONEXPLICIT_AFFECTION_ACTIVITIES = SOFT_AFFECTION_ACTIVITIES

COMMITMENT_ACTS = ["vowing", "affirming_relationship", "confessing_love", "love_confession"]
COMMITMENT_SETTINGS = ["wedding_planning", "wedding_aisle"]

RITUAL_MARKERS = [
    "gift_giving", "planning_surprise", "celebrating", "dancing",
    "kneeling", "vowing", "confessing_love", "love_confession",
    "affirming_relationship"
]
RITUAL_SETTINGS = ["wedding_planning", "wedding_aisle", "party", "gathering", "parade", "church"]

MISCOMM_STRONG = ["lying", "rejecting", "silent_staring", "checking_phone", "texting", "arguing", "argument", "threatening"]
REPAIR_ACTS = ["apologizing", "saying_sorry", "regret_expression", "affirmation", "affirming_relationship",
               "confession", "confessing", "confessing_love", "love_confession", "vowing", "thanking", "expressing_gratitude", "reunion"]

VICE_ACTS = ["drinking_wine", "wine_tasting", "partying"]
VICE_SETTINGS = ["bar", "barroom", "club", "disco", "distillery"]

HEALTH_ACTS = ["medical_emergency", "accident", "awaiting_results"]
HEALTH_SETTINGS = ["hospital", "clinic"]

APPEARANCE_ACTS = ["grooming", "posing", "admiring", "seductive_modeling"]

TECH_SETTINGS = ["laptop_or_phone", "webcam", "phone"]
TECH_ACTS = ["checking_phone", "video_chat"]

print("✓ Tag lexicons defined")


✓ Tag lexicons defined


In [8]:
# -----------------------------
# Taxonomy-first composite definitions
# -----------------------------
# First, get available subgroups from topic_spine (will be populated after data loading)
# For now, define the structure - we'll resolve subgroup names after loading

# This will be populated after topic_spine is loaded
available_subgroups = []

def sg(names: list) -> list:
    """Helper to resolve subgroup names with suggestions for missing ones."""
    if not available_subgroups:
        return names  # Will be resolved later
    resolved, missing = resolve_subgroups(names, available_subgroups)
    if missing:
        print("\n⚠️  Missing subgroup names (showing suggestions):")
        for req, sugg in missing:
            print(f"  - '{req}' -> suggestions: {sugg}")
    return resolved

# Core subgroup lists (taxonomy spine) - will be resolved after data load
CORE = {
    "A_reassurance_commitment": ["Relationship Stage & Commitment", "Rupture, Separation & Reconciliation"],
    "B_mutual_intimacy": ["Soft Affection & Non-Sexual Touch", "Sexual Arousal & Foreplay"],
    "C_explicit": ["Explicit Sexual Acts"],
    "D_luxury_status": ["Money, Wealth & Economic Security", "Luxury Lifestyle & Status Performance", "Work & Professional Life"],
    "E_coercion_brutality_danger": ["Physical Threats & Violence", "Psychological Harm & Trauma"],
    "F_angst_negative_affect": ["Vulnerability, Sadness & Fear", "Anger, Resentment & Hostility", "Guilt, Shame & Moral Conflict", "Inner Conflict, Decisions & Reflection"],
    "G_courtship_rituals_gifts": ["Courtship Rituals & Romantic Gestures", "Time & Life Events"],
    "H_domestic_nesting": ["Domestic Spaces & Home Life"],
    "I_humor_lightness": ["Positive Emotions & Safety"],
    "J_social_support_kin": ["Family & Kinship", "Children & Parenthood", "Friends, Colleagues & Community"],
    "K_professional_intrusion": ["Work & Professional Life"],
    "L_vices_addictions": ["Addictions & Risky Behaviours"],
    "M_health_recovery_growth": ["Health, Care & Recovery", "Memory, Learning & Personal Growth"],
    "N_separation_reunion": ["Rupture, Separation & Reconciliation"],
    "O_aesthetics_appearance": ["Appearance, Clothing & Grooming", "Sensory Impressions"],
    "P_tech_media_presence": ["Technology, Media & Art"],
    "Q_miscommunication": ["Communication & Miscommunication"],
    "Q_repair": ["Rupture, Separation & Reconciliation", "Relationship Stage & Commitment"],
    "R_protectiveness": ["Health, Care & Recovery", "Physical Threats & Violence"],
    "R_jealousy": ["Interpersonal Conflict & Betrayal", "Social Roles & Power/Control", "Communication & Miscommunication"],
}

print("✓ Core taxonomy subgroups defined (will be resolved after data load)")


✓ Core taxonomy subgroups defined (will be resolved after data load)


## 6. Composite Definitions

Define composite indices using the extended rule engine. **D_luxury_status** uses `include_all` to require both status-coded setting AND status-coded activity.


In [9]:
COMPOSITES = {
    # A) Reassurance / Commitment (HEA Centrality)
    "A_reassurance_commitment": {
        "include_any": [
            {"taxonomy_subgroups": ["Reconciliation, Commitments & HEA"]},
            {"primary_any": ["romance_core"]},
            {"keyword_any": ["apology", "apologize", "forgive", "forgiveness", "commitment", "promise", "vow", "marriage", "engagement", "ring", "wedding"]},
        ],
    },
    
    # B) Mutual Intimacy (Non-Explicit)
    "B_mutual_intimacy": {
        "include_any": [
            {"taxonomy_subgroups": ["Kissing & Non-Explicit Affection"]},
            {"primary_any": ["physical_affection"]},
            {"secondary_dim_any": {"activity": NONEXPLICIT_AFFECTION_ACTIVITIES}},
        ],
    },
    
    # C) Explicit Eroticism
    "C_explicit": {
        "include_any": [
            {"taxonomy_subgroups": ["Explicit Sexual Acts"]},
            {"primary_any": ["sexual_content"]},
            {"secondary_dim_any": {"activity": EXPLICIT_ACTIVITIES}},
        ],
        "exclude_any": [
            # Remove topics that are mostly "consent talk/aftercare" if they exist
            {"keyword_any": ["consent", "aftercare", "post-sex"]},
        ],
    },
    
    # D) Power / Wealth / Luxury
    "D_luxury_status": {
        "include_any": [
            {"taxonomy_subgroups": ["Hero's Elite Work & Business World", "Money, Housing & Economic Security"]},
            {"primary_any": ["business_setting", "business_or_work"]},
        ],
        "include_all": [
            # 1) status-coded setting OR luxury keywords
            {
                "secondary_dim_any": {"setting": ["boardroom", "press_conference", "hotel_suite", "airplane", "boat", "cockpit", "pool", "club", "gallery", "home_office"]},
                "keyword_any": ["penthouse", "private jet", "designer", "chauffeur", "paparazzi", "yacht", "limousine"],
            },
            # 2) status-coded activity OR luxury keywords
            {
                "secondary_dim_any": {"activity": ["shopping", "wine_tasting", "partying", "photography"]},
                "keyword_any": ["designer", "penthouse", "private jet", "paparazzi", "chauffeur", "yacht"],
            },
        ],
        "exclude_any": [
            # remove plain school/work if it slips in
            {"primary_any": ["work_or_school"]},
        ],
    },
    
    # E) Coercion / Brutality / Danger (Dark Themes)
    "E_coercion_brutality_danger": {
        "include_any": [
            {"taxonomy_subgroups": ["Violence, Threats & Coercion"]},
            {"keyword_any": ["brutal", "danger", "coercion", "weapon", "security", "revenge", "jail", "torture", "violence", "threat", "threaten"]},
        ],
    },
    
    # F) Angst / Negative Affect
    "F_angst_negative_affect": {
        "include_any": [
            {"taxonomy_subgroups": ["Negative Emotions & Distress", "Conflict, Distance & Breakup Threats"]},
            {"primary_any": ["relationship_conflict"]},
            {"keyword_any": ["conflict", "doubt", "anxiety", "jealousy", "guilt", "betrayal", "crying", "tears", "angry", "resentment"]},
        ],
    },
    
    # G) Courtship Rituals / Gifts (HEA Component)
    "G_courtship_rituals_gifts": {
        "include_any": [
            {"taxonomy_subgroups": ["Public & Leisure Spaces"]},
            {"keyword_any": ["courtship", "date", "gift", "proposal", "wedding", "dance", "ritual", "jewelry", "ring", "dinner", "restaurant"]},
        ],
    },
    
    # H) Domestic Nesting (Home-as-Refuge)
    "H_domestic_nesting": {
        "include_any": [
            {"taxonomy_subgroups": ["Domestic Spaces & Routines"]},
            {"keyword_any": ["domestic", "home", "bedroom", "kitchen", "house", "cozy", "cooking", "meal"]},
        ],
    },
    
    # I) Humor / Lightness
    "I_humor_lightness": {
        "include_any": [
            {"taxonomy_subgroups": ["Positive Emotions & Security"]},
            {"keyword_any": ["laughter", "laugh", "joy", "humor", "banter", "playful", "smile", "grin", "giggle"]},
        ],
    },
    
    # J) Social Support / Kin
    "J_social_support_kin": {
        "include_any": [
            {"taxonomy_subgroups": ["Family & Kinship", "Friends & Social Circles"]},
            {"keyword_any": ["family", "sibling", "brother", "sister", "parent", "friend", "friends", "community", "social"]},
        ],
    },
    
    # K) Professional Intrusion (Office/Corporate Frame Share)
    "K_professional_intrusion": {
        "include_any": [
            {"taxonomy_subgroups": ["Hero's Elite Work & Business World"]},
            {"primary_any": ["work_or_school"]},
            {"keyword_any": ["meeting", "office", "appointment", "schedule", "time", "work", "job", "boss", "corporate", "business"]},
        ],
    },
    
    # L) Vices / Addictions
    "L_vices_addictions": {
        "include_any": [
            {"keyword_any": ["alcohol", "addiction", "drug", "vice", "nightlife", "party", "drunk", "drinking", "bar", "club"]},
        ],
    },
    
    # M) Health / Recovery / Growth (Tender Care + Healing Arcs)
    "M_health_recovery_growth": {
        "include_any": [
            {"keyword_any": ["health", "medical", "recovery", "therapy", "growth", "development", "fertility", "pregnancy", "baby", "healing", "care"]},
        ],
    },
    
    # N) Separation / Reunion (Arc Mechanics)
    "N_separation_reunion": {
        "include_any": [
            {"taxonomy_subgroups": ["Reconciliation, Commitments & HEA"]},
            {"keyword_any": ["separation", "goodbye", "reunion", "return", "leave", "leaving", "depart", "arrive", "back"]},
        ],
    },
    
    # O) Aesthetics / Appearance (Visual/Cultural Cues)
    "O_aesthetics_appearance": {
        "include_any": [
            {"keyword_any": ["clothes", "appearance", "makeup", "jewelry", "underwear", "fashion", "style", "dress", "outfit", "wardrobe", "grooming"]},
        ],
    },
    
    # P) Tech / Media Presence
    "P_tech_media_presence": {
        "include_any": [
            {"keyword_any": ["tech", "phone", "text", "media", "paparazzi", "news", "report", "public image", "scandal", "screen", "message", "notification"]},
        ],
    },
    
    # Q) Miscommunication vs Repair (Balance Index)
    # Q_miscommunication: Communication conflicts
    "Q_miscommunication": {
        "include_any": [
            {"taxonomy_subgroups": ["Secrets, Misunderstandings & Hidden Information"]},
            {"keyword_any": ["miscommunication", "misunderstanding", "secret", "lie", "deception", "argument", "quarrel", "silence"]},
        ],
    },
    
    # Q_repair: Apologies, forgiveness, reconciliation
    "Q_repair": {
        "include_any": [
            {"taxonomy_subgroups": ["Reconciliation, Commitments & HEA"]},
            {"keyword_any": ["apology", "apologize", "forgive", "forgiveness", "reconciliation", "repair", "fix", "resolve"]},
        ],
    },
    
    # R) Protectiveness vs Jealousy (Delta Index)
    # R_protectiveness: Caring protectiveness
    "R_protectiveness": {
        "include_any": [
            {"keyword_any": ["protect", "protection", "protective", "care", "caring", "safe", "safety", "guard", "shield", "support"]},
        ],
    },
    
    # R_jealousy: Jealous/possessive behavior
    "R_jealousy": {
        "include_any": [
            {"keyword_any": ["jealous", "jealousy", "possessive", "possess", "ownership", "claim", "mine", "territory"]},
        ],
    },
}

print("✓ Composite definitions loaded")
print(f"  Composites: {list(COMPOSITES.keys())}")


✓ Composite definitions loaded
  Composites: ['A_reassurance_commitment', 'B_mutual_intimacy', 'C_explicit', 'D_luxury_status', 'E_coercion_brutality_danger', 'F_angst_negative_affect', 'G_courtship_rituals_gifts', 'H_domestic_nesting', 'I_humor_lightness', 'J_social_support_kin', 'K_professional_intrusion', 'L_vices_addictions', 'M_health_recovery_growth', 'N_separation_reunion', 'O_aesthetics_appearance', 'P_tech_media_presence', 'Q_miscommunication', 'Q_repair', 'R_protectiveness', 'R_jealousy']


In [10]:
# --------------------------
# Tag lexicons tuned to your tagger output
# --------------------------

EXPLICIT_ACTIVITIES = [
    # clearly explicit / sexual mechanics
    "oral_sex", "mutual_stimulation", "nipple_play", "BDSM",
    "checking_condom", "comparing_sizes",
]
EXPLICIT_SEXUAL_DIM = [
    # sexual:* dim values (from your unique list)
    "oral_sex", "clitoral_stimulation", "waves_of_orgasm",
    "nudity", "hair_pulling", "compromising_positions", "dominatrix", "touching", "affair",
]

NONEXPLICIT_AFFECTION_ACTIVITIES = [
    "kissing", "making_out", "flirting", "gazing", "eye_contact",
    "smiling", "laughing", "teasing", "knowing_glance", "whispering",
    "back_to_back", "staring_and_smiling",
]

RITUALS_GIFTS_ACTIVITIES = [
    "gift_giving", "planning_surprise", "celebrating", "dancing", "invitation",
    "dinner", "wine_tasting", "kneeling", "vowing", "confessing_love",
    "love_confession", "affirming_relationship",
]

REPAIR_ACTIVITIES = [
    "apologizing", "saying_sorry", "regret_expression",
    "affirmation", "affirming_relationship",
    "confession", "confessing", "confessing_love", "love_confession",
    "vowing", "offering", "thanking", "expressing_gratitude",
    "sharing_good_news", "reunion",
]

MISCOMM_ACTIVITIES = [
    "arguing", "argument", "lying", "rejecting",
    "texting", "checking_phone", "silent", "silent_staring",
    "threatening", "warning", "questioning",
]

DANGER_ACTIVITIES = [
    "threatening", "warning", "calling_police", "threat_assessment", "planning_murder",
    "investigation", "intervening",
]

HUMOR_ACTIVITIES = ["laughing", "sarcasm", "teasing", "horseplay", "cheering"]

VICE_ACTIVITIES = ["drinking_wine", "wine_tasting", "partying"]
VICE_SETTINGS = ["bar", "barroom", "club", "disco", "distillery"]

HEALTH_SETTINGS = ["hospital", "clinic", "police_station"]  # police_station sometimes threat-related; keep in E/K as needed
HEALTH_ACTIVITIES = ["medical_emergency", "accident", "awaiting_results"]
HEALTH_OBJECTS = []  # your tagger doesn't seem to emit object:* currently; keep placeholder

APPEARANCE_ACTIVITIES = ["grooming", "posing", "admiring", "seductive_modeling"]
APPEARANCE_SETTINGS = ["clothing_store", "gallery"]  # gallery is shaky; keep only if it codes aesthetics

LUXURY_PROXY_SETTINGS = [
    # status-coded settings you actually have
    "boardroom", "press_conference", "hotel_suite", "home_office",
    "cockpit", "airplane", "boat", "pool", "club", "gallery",
]
LUXURY_PROXY_ACTIVITIES = ["shopping", "wine_tasting", "partying", "photography"]

# --------------------------
# Composite definitions (A–S) using your real primary + secondary vocab
# --------------------------

COMPOSITES = {

    # A: Commitment / HEA (strongly tied to commitment acts; plus wedding planning)
    "A_commitment_hea": {
        "include": {
            # keep subgroup backbone if you have it in topic_lookup
            "taxonomy_subgroups": ["Reconciliation, Commitments & HEA"],
            # tag-driven: explicit relationship affirmation
            "secondary_dim_any": {
                "activity": ["vowing", "affirming_relationship", "confessing_love", "love_confession"],
                "setting": ["wedding_planning", "wedding_aisle"],
                "relationship": ["long_term"],
            },
            # primary categories that often mark endgame stability (use carefully)
            "primary_any": ["romance_core"],
        },
        "exclude": {
            # keep explicit mechanics out of A
            "secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM, "activity": EXPLICIT_ACTIVITIES},
            "primary_any": ["nonfiction_or_technical"],
        },
    },

    # B: Mutual intimacy (non-explicit; allow kissing/making_out; exclude explicit)
    "B_mutual_intimacy": {
        "include": {
            "taxonomy_subgroups": ["Kissing & Non-Explicit Affection", "Attraction & Sexual Tension", "Bonding, Everyday Intimacy & Growth"],
            "primary_any": ["physical_affection", "sexual_tension", "romance_core"],
            "secondary_dim_any": {"activity": NONEXPLICIT_AFFECTION_ACTIVITIES},
        },
        "exclude": {
            "secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM, "activity": EXPLICIT_ACTIVITIES},
            # if you want B to be "love not sex", exclude high sexual_content topics unless they are only kissing/making_out
            # comment out if too aggressive:
            # "primary_any": ["sexual_content"],
        },
    },

    # C: Explicit erotics (sexual_content + explicit signals)
    "C_explicit": {
        "include": {
            "primary_any": ["sexual_content"],
            "secondary_dim_any": {
                "activity": EXPLICIT_ACTIVITIES + ["foreplay"],  # foreplay often sits on the boundary; keep here if you want "sex-heavy" indexing
                "sexual": EXPLICIT_SEXUAL_DIM,
            },
        },
        "exclude": {
            # remove "just kissing" topics that got sexual_content label
            "secondary_dim_any": {"activity": ["kissing", "making_out"]},
        },
    },

    # D: Luxury / status (proxy using available settings + activities)
    "D_luxury_status": {
        "include": {
            # if you have Work/Wealth taxonomy buckets, keep them as anchor:
            "taxonomy_subgroups": ["Hero's Elite Work & Business World", "Money, Housing & Economic Security"],
            "primary_any": ["business_setting", "business_or_work"],
            "secondary_dim_any": {"setting": LUXURY_PROXY_SETTINGS, "activity": LUXURY_PROXY_ACTIVITIES},
        },
        "exclude": {
            # keep plain school/work without status cues out if you find D too broad
            "primary_any": ["work_or_school"],
        }
    },

    # E: Threat / coercion / danger (tagger has strong danger-like activities)
    "E_threat_danger": {
        "include": {
            "taxonomy_subgroups": ["Violence, Threats & Coercion"],
            "secondary_dim_any": {"activity": DANGER_ACTIVITIES, "setting": ["police_station", "warehouse", "desolate"]},
            "primary_any": ["relationship_conflict"],  # optional; drop if too broad
            "keyword_any": ["gun", "knife", "kidnap", "threat", "coerc", "violence"],
        },
        "exclude": {
            # keep normal arguing from becoming "danger"
            "secondary_dim_any": {"activity": ["arguing", "argument"]},
        },
    },

    # F: Angst / negative affect (you don't have emotion:* dim; infer via primary + conflict activities)
    "F_angst_negative": {
        "include": {
            "primary_any": ["emotional_content", "relationship_conflict"],
            "secondary_dim_any": {"activity": ["crying", "worrying", "struggling", "whining", "scowling", "frowning"] + MISCOMM_ACTIVITIES},
            "taxonomy_subgroups": ["Negative Emotions & Distress", "Ambivalence & Internal Conflict", "Conflict, Distance & Breakup Threats"],
        },
        "exclude": {
            # keep external danger separate
            "secondary_dim_any": {"activity": DANGER_ACTIVITIES},
        },
    },

    # G: Rituals / gifts / courtship gestures (your tagger supports gift_giving etc)
    "G_rituals_gifts": {
        "include": {
            "primary_any": ["romance_core", "social_setting", "domestic_life"],
            "secondary_dim_any": {
                "activity": RITUALS_GIFTS_ACTIVITIES,
                "setting": ["restaurant", "party", "gathering", "parade", "church", "wedding_planning", "wedding_aisle"],
            },
        },
        "exclude": {
            # keep explicit sex from inflating rituals
            "secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM, "activity": EXPLICIT_ACTIVITIES},
        },
    },

    # H: Domestic nesting (domestic_life + home-ish settings + routines)
    "H_domestic_nesting": {
        "include": {
            "primary_any": ["domestic_life"],
            "secondary_dim_any": {"setting": ["home", "apartment", "living_room", "kitchen", "bathroom"], "activity": ["preparing", "sleeping", "bedtime", "relaxing"]},
            "taxonomy_subgroups": ["Domestic Spaces & Routines"],
        },
        "exclude": {
            # optionally remove bedroom-heavy sex topics
            "secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM, "activity": EXPLICIT_ACTIVITIES},
        },
    },

    # I: Humor / lightness (your tagger has sarcasm/teasing/laughing)
    "I_humor_lightness": {
        "include": {"secondary_dim_any": {"activity": HUMOR_ACTIVITIES}},
        "exclude": {"primary_any": ["nonfiction_or_technical"]},
    },

    # J: Social support / kin (you have relationship:family; plus parenting; plus social_setting)
    "J_social_support": {
        "include": {
            "primary_any": ["social_setting"],
            "secondary_dim_any": {"relationship": ["family"], "activity": ["parenting", "visiting", "celebrating", "gathering"]},
            "taxonomy_subgroups": ["Family & Kinship", "Friends & Social Circles", "Community, Norms & Social Events"],
        }
    },

    # K: Professional intrusion / workplace frame (business/work primaries + office/boardroom etc)
    "K_professional_frame": {
        "include": {
            "primary_any": ["business_or_work", "business_setting", "work_or_school"],
            "secondary_dim_any": {"setting": ["office", "boardroom", "school", "seminar", "press_conference", "home_office"], "activity": ["working", "discussing_business", "negotiating", "questioning"]},
            "taxonomy_subgroups": ["Shared Workplaces & Professional Interaction", "Heroine's Work & Professional Identity", "Hero's Elite Work & Business World"],
        }
    },

    # L: Vices / risky behaviors (your tagger has drinking_wine/wine_tasting/partying + bar/club)
    "L_vices": {
        "include": {
            "secondary_dim_any": {"activity": VICE_ACTIVITIES, "setting": VICE_SETTINGS},
            "keyword_any": ["cigarette", "drug", "vodka", "whiskey", "coke", "pill", "rehab"],  # fallback
        },
        "exclude": {
            # keep romantic dinners from being treated as "vices" if you want
            "secondary_dim_any": {"activity": ["dinner", "invitation"]},
        }
    },

    # M: Health / recovery / crisis-care
    "M_health_recovery": {
        "include": {
            "secondary_dim_any": {"setting": HEALTH_SETTINGS, "activity": HEALTH_ACTIVITIES},
            "keyword_any": ["hospital", "doctor", "nurse", "therapy", "injury", "healing", "recovery"],
        }
    },

    # O: Appearance / grooming / aesthetics
    "O_appearance": {
        "include": {
            "secondary_dim_any": {"activity": APPEARANCE_ACTIVITIES, "setting": APPEARANCE_SETTINGS},
            "keyword_any": ["dress", "heels", "lipstick", "perfume", "suit", "tie", "lingerie", "makeup", "hair"],
        },
        "exclude": {
            # nudity is erotic explicitness, not aesthetics
            "secondary_dim_any": {"sexual": ["nudity"]},
        }
    },

    # Q: Miscommunication vs Repair (split; your tags support both well)
    "Q_miscommunication": {
        "include": {
            "primary_any": ["relationship_conflict"],
            "secondary_dim_any": {"activity": MISCOMM_ACTIVITIES + ["discussion", "discussing"]},
            "taxonomy_subgroups": ["Secrets, Misunderstandings & Hidden Information"],
        },
        "exclude": {
            # keep danger-related content out
            "secondary_dim_any": {"activity": DANGER_ACTIVITIES},
        }
    },

    "Q_repair": {
        "include": {
            "secondary_dim_any": {"activity": REPAIR_ACTIVITIES},
            "taxonomy_subgroups": ["Reconciliation, Commitments & HEA"],
        },
        "exclude": {
            "secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM, "activity": EXPLICIT_ACTIVITIES},
        }
    },

    # R: Protectiveness vs Jealousy
    # Protectiveness: intervention + safety + medical emergency cues
    "R_protectiveness": {
        "include": {
            "secondary_dim_any": {"activity": ["intervening", "warning", "calling_police", "offering", "preparing", "threat_assessment"] + HEALTH_ACTIVITIES},
            "keyword_any": ["protect", "safe", "shield", "guard", "security", "keep you safe"],
        }
    },

    # Jealousy: you don't have an explicit jealousy tag, so lean on keywords + phone-checking + rivalry-ish behaviors
    "R_jealousy": {
        "include": {
            "secondary_dim_any": {"activity": ["checking_phone", "questioning", "arguing", "threatening"]},
            "keyword_any": ["jealous", "possess", "mine", "don't talk to", "belongs to me"],
        }
    },

    # S: Scene anchors (sampling tool): setting-rich topics
    "S_scene_anchors": {
        "include": {
            "secondary_dim_any": {"setting": [
                "home", "apartment", "living_room", "kitchen", "office", "boardroom",
                "restaurant", "bar", "club", "hotel_suite", "beach", "pool", "airplane", "car"
            ]},
            "primary_any": ["domestic_life", "social_setting", "business_setting"],
        },
        "exclude": {"primary_any": ["nonfiction_or_technical"]}
    },
}

print("✓ Composite definitions loaded")
print(f"  Composites: {list(COMPOSITES.keys())}")


✓ Composite definitions loaded
  Composites: ['A_commitment_hea', 'B_mutual_intimacy', 'C_explicit', 'D_luxury_status', 'E_threat_danger', 'F_angst_negative', 'G_rituals_gifts', 'H_domestic_nesting', 'I_humor_lightness', 'J_social_support', 'K_professional_frame', 'L_vices', 'M_health_recovery', 'O_appearance', 'Q_miscommunication', 'Q_repair', 'R_protectiveness', 'R_jealousy', 'S_scene_anchors']


## 7. Load Data & Prepare Topic Spine

Load topic lookup and taxonomy mapping to build the topic spine with parsed categories.


## 7.5. Diagnostic: Verify Data Parsing & Weights

Quick sanity checks to ensure parsing worked and weights are computed correctly.

**Note:** Run cell 23 (Load Data & Prepare Topic Spine) first before running this diagnostic.


In [11]:
# --- quick sanity checks: do we actually have parsed tokens? ---

try:
    print("topic_spine rows:", len(topic_spine))

    # pick a row that has non-null categories
    ex = topic_spine.loc[
        topic_spine["primary_categories"].notna() & topic_spine["secondary_categories"].notna()
    ].head(3)
    display(
        ex[
            [
                "topic_id",
                "primary_categories",
                "secondary_categories",
                "primary_set",
                "secondary_set",
                "secondary_dim",
            ]
        ]
    )

    # how many topics have at least one primary/secondary token?
    print(
        "non-empty primary_set:",
        int(topic_spine["primary_set"].apply(len).gt(0).sum()),
    )
    print(
        "non-empty secondary_set:",
        int(topic_spine["secondary_set"].apply(len).gt(0).sum()),
    )

    # check gate/weights if you already computed them
    if "w_eff" in topic_spine.columns:
        print(
            "w_eff > 0:",
            int((topic_spine["w_eff"] > 0).sum()),
            "out of",
            len(topic_spine),
        )
        print(topic_spine["w_eff"].describe())
except NameError as e:
    print(f"Sanity check skipped: {e}")



Sanity check skipped: name 'topic_spine' is not defined


## 7.6. Safe Weight Computation

Compute effective weights with safe handling of NaN gate3_pass values.


In [12]:
def compute_w_eff_safe(df: pd.DataFrame) -> pd.Series:
    """
    Compute effective weights with safe handling of NaN gate3_pass values.
    Key change: NaN gate3_pass is treated as pass, not fail.
    """
    w = np.ones(len(df), dtype=float)
    
    # Hard noise exclusion
    exclude = df["label_is_noise"].fillna(False).astype(bool) if "label_is_noise" in df.columns else pd.Series(False, index=df.index)
    
    # Also check taxonomy_is_noise if present
    if "taxonomy_is_noise" in df.columns:
        exclude |= df["taxonomy_is_noise"].fillna(False).astype(bool)
    
    # Gate3 pass: only apply if the column has *some* non-null values
    if "gate3_pass" in df.columns:
        gp = df["gate3_pass"]
        if gp.notna().any():
            exclude |= (~gp.fillna(True).astype(bool))  # NaN -> True (don't exclude unknowns)
    
    # Optional downweights
    if "taxonomy_confidence" in df.columns:
        conf = df["taxonomy_confidence"].fillna("").astype(str).str.lower()
        w *= np.where(conf.eq("medium"), 0.6, 1.0)
        w *= np.where(conf.eq("low"), 0.4, 1.0)
    
    if "radway_confidence" in df.columns:
        rconf = df["radway_confidence"].fillna("").astype(str).str.lower()
        w *= np.where(rconf.eq("medium"), 0.85, 1.0)
        w *= np.where(rconf.eq("low"), 0.70, 1.0)
    
    w = np.where(exclude, 0.0, w)
    return pd.Series(w, index=df.index)

print("✓ Weight computation function defined (will be used after topic_spine is created)")


✓ Weight computation function defined (will be used after topic_spine is created)


In [13]:
# Load topic lookup
topic_lookup = pd.read_parquet(TOPIC_LOOKUP_PATH)
topic_lookup["topic_id"] = topic_lookup["topic_id"].astype(int)

# Load topic taxonomy mapping
topic_tax_map = pd.read_csv(TOPIC_TAXONOMY_MAPPING_PATH)
topic_tax_map["topic_id"] = topic_tax_map["topic_id"].astype(int)

# Merge taxonomy info (keep both _x and _y suffixes for now, we'll standardize)
topic_spine = topic_lookup.merge(topic_tax_map, on="topic_id", how="left", suffixes=("_lookup", "_taxmap"))

# Parse primary / secondary tags into sets using new parsing functions
topic_spine["primary_set"] = topic_spine.get("primary_categories", pd.Series([None]*len(topic_spine))).apply(parse_cat_field)
topic_spine["secondary_set"] = topic_spine.get("secondary_categories", pd.Series([None]*len(topic_spine))).apply(parse_cat_field)
topic_spine["secondary_dim"] = topic_spine["secondary_set"].apply(secondary_dim_map)

# Standardize taxonomy columns so rule engine sees them
# Prefer the mapping file (taxmap) if present, else lookup.
if "taxonomy_main_name" not in topic_spine.columns:
    for cand in ["taxonomy_main_name_taxmap", "taxonomy_main_name_lookup", "taxonomy_main_name_x", "taxonomy_main_name_y"]:
        if cand in topic_spine.columns:
            topic_spine["taxonomy_main_name"] = topic_spine[cand]
            break

if "taxonomy_main_group" not in topic_spine.columns:
    for cand in ["taxonomy_main_group_taxmap", "taxonomy_main_group_lookup", "taxonomy_main_group_x", "taxonomy_main_group_y"]:
        if cand in topic_spine.columns:
            topic_spine["taxonomy_main_group"] = topic_spine[cand]
            break

# Join topic_health for prevalence / mass if available
if TOPIC_HEALTH_PATH.exists():
    topic_health = pd.read_parquet(TOPIC_HEALTH_PATH)
    topic_health["topic_id"] = topic_health["topic_id"].astype(int)
    keep_cols = [c for c in ["topic_id","prevalence","mass","gate3_pass"] if c in topic_health.columns]
    topic_spine = topic_spine.merge(topic_health[keep_cols], on="topic_id", how="left")
else:
    topic_spine["prevalence"] = 1.0

# Compute weights using safe function
topic_spine["w_eff"] = compute_w_eff_safe(topic_spine)

print("✓ Topic spine prepared:", topic_spine.shape)
print("  non-empty primary_set:", int(topic_spine["primary_set"].apply(len).gt(0).sum()))
print("  non-empty secondary_set:", int(topic_spine["secondary_set"].apply(len).gt(0).sum()))
print("  w_eff > 0:", int((topic_spine["w_eff"] > 0).sum()), "out of", len(topic_spine))


✓ Topic spine prepared: (369, 55)
  non-empty primary_set: 351
  non-empty secondary_set: 351
  w_eff > 0: 256 out of 369


## 7.7. FIX: Standardize Taxonomy Columns (Main Groups vs Subgroups)

**Critical fix:** The 28-node taxonomy subgroups live in `taxonomy_secondary_name`, not `taxonomy_main_name`.
- `taxonomy_main_name` = 7/8 main groups
- `taxonomy_secondary_name` = 28 subgroups (the actual spine for composites)


In [14]:
# ============================================================
# FIX: Use taxonomy_secondary_name as the 28-subgroup spine
#      and taxonomy_main_name as the 7/8 main-group spine.
# Also updates the rule engine to match against the right columns.
# ============================================================

def _pick_first_nonnull(df, candidates, newcol):
    for c in candidates:
        if c in df.columns:
            s = df[c]
            if s.notna().any():
                df[newcol] = s
                return
    # fallback
    df[newcol] = np.nan

# Main group (7/8): use taxonomy_main_name / taxonomy_main_group
_pick_first_nonnull(
    topic_spine,
    ["taxonomy_main_name", "taxonomy_main_name_x", "taxonomy_main_name_y",
     "taxonomy_main_name_lookup", "taxonomy_main_name_taxmap"],
    "tax_main_name"
)

_pick_first_nonnull(
    topic_spine,
    ["taxonomy_main_group", "taxonomy_main_group_x", "taxonomy_main_group_y",
     "taxonomy_main_group_lookup", "taxonomy_main_group_taxmap"],
    "tax_main_group"
)

_pick_first_nonnull(
    topic_spine,
    ["taxonomy_main_id", "taxonomy_main_id_lookup", "taxonomy_main_id_taxmap"],
    "tax_main_id"
)

# Subgroup (28): use taxonomy_secondary_name / taxonomy_secondary_group / taxonomy_secondary_id
_pick_first_nonnull(
    topic_spine,
    ["taxonomy_secondary_name", "taxonomy_secondary_name_lookup", "taxonomy_secondary_name_taxmap"],
    "tax_sub_name"
)

_pick_first_nonnull(
    topic_spine,
    ["taxonomy_secondary_group", "taxonomy_secondary_group_lookup", "taxonomy_secondary_group_taxmap"],
    "tax_sub_group"
)

_pick_first_nonnull(
    topic_spine,
    ["taxonomy_secondary_id", "taxonomy_secondary_id_lookup", "taxonomy_secondary_id_taxmap"],
    "tax_sub_id"
)

print("✓ Taxonomy columns standardized")
print("  main groups non-null:", int(topic_spine["tax_main_name"].notna().sum()))
print("  subgroups non-null:", int(topic_spine["tax_sub_name"].notna().sum()))
print("\nUnique main groups (sample):", sorted(topic_spine["tax_main_name"].dropna().astype(str).unique())[:10])
print("Unique subgroups (sample):", sorted(topic_spine["tax_sub_name"].dropna().astype(str).unique())[:15])


✓ Taxonomy columns standardized
  main groups non-null: 361
  subgroups non-null: 59

Unique main groups (sample): ['Ambivalence & Internal Conflict', 'Attraction & Sexual Tension', 'Beliefs, Values & Moral Reflection', 'Body Parts & Physical Reactions', 'Bonding, Everyday Intimacy & Growth', 'Community, Norms & Social Events', 'Conflict, Distance & Breakup Threats', 'Domestic Spaces & Routines', 'Exercise & Physical Activity', 'Explicit Sexual Acts']
Unique subgroups (sample): ['Ambivalence & Internal Conflict', 'Attraction & Sexual Tension', 'Body Parts & Physical Reactions', 'Bonding, Everyday Intimacy & Growth', 'Conflict, Distance & Breakup Threats', 'Domestic Spaces & Routines', 'Exercise & Physical Activity', 'Friends & Social Circles', "Hero's Elite Work & Business World", "Heroine's Work & Professional Identity", 'Meeting, First Impressions & Setup', 'Money, Housing & Economic Security', 'Negative Emotions & Distress', 'Positive Emotions & Security', 'Public & Leisure Spaces']

## 7.7.1. FIX: Unified Taxonomy Node Column (Handles Sparse Secondary)

**Issue:** `taxonomy_secondary_name` only exists for 59 topics, but `taxonomy_main_name` contains the 28-node taxonomy for 361 topics.

**Solution:** Create unified `tax_node_name = tax_sub_name if present else tax_main_name` to use as the subgroup spine.


In [15]:
# ============================================================
# FIX: sparse taxonomy_secondary_name (only 59 topics)
# -> use a unified taxonomy "node" column for subgroup spine
# ============================================================

# 1) Build unified node + group columns
# - tax_main_name: appears to be your 28-node taxonomy for most topics (361 non-null)
# - tax_sub_name: only populated for 59 topics (too sparse to use alone)
topic_spine["tax_node_name"] = topic_spine["tax_sub_name"].fillna(topic_spine["tax_main_name"])
topic_spine["tax_group_name"] = topic_spine["tax_main_group"]  # keep as main group spine (7/8 groups)

print("✓ Unified taxonomy spine built")
print("  tax_node_name non-null:", int(topic_spine["tax_node_name"].notna().sum()))
print("  unique tax_node_name:", len(topic_spine["tax_node_name"].dropna().astype(str).unique()))
print("  tax_group_name non-null:", int(topic_spine["tax_group_name"].notna().sum()))
print("\nSample tax_node_name:", sorted(topic_spine["tax_node_name"].dropna().astype(str).unique())[:20])
print("Sample tax_group_name:", sorted(topic_spine["tax_group_name"].dropna().astype(str).unique())[:10])


✓ Unified taxonomy spine built
  tax_node_name non-null: 361
  unique tax_node_name: 28
  tax_group_name non-null: 361

Sample tax_node_name: ['Ambivalence & Internal Conflict', 'Attraction & Sexual Tension', 'Beliefs, Values & Moral Reflection', 'Body Parts & Physical Reactions', 'Bonding, Everyday Intimacy & Growth', 'Community, Norms & Social Events', 'Conflict, Distance & Breakup Threats', 'Domestic Spaces & Routines', 'Exercise & Physical Activity', 'Explicit Sexual Acts', 'Family & Kinship', 'Friends & Social Circles', "Hero's Elite Work & Business World", "Heroine's Work & Professional Identity", 'Interpersonal Non-Romantic Conflict', 'Kissing & Non-Explicit Affection', 'Law, Medicine, Education & Formal Institutions', 'Meeting, First Impressions & Setup', 'Money, Housing & Economic Security', 'Negative Emotions & Distress']
Sample tax_group_name: ['Conflict, Risk & Harm', 'Embodied & Sensory Experience', 'Emotions, Cognition & Inner Life', 'Relationship Trajectory (Main Couple)

In [16]:
# -----------------------------
# Rebuild available_subgroups from tax_node_name (unified, NOT tax_sub_name alone)
# -----------------------------
available_subgroups = sorted(topic_spine["tax_node_name"].dropna().astype(str).unique().tolist())
print("\n✓ Available subgroup nodes for CORE matching:", len(available_subgroups))
print("  Full list:")
for i, sg in enumerate(available_subgroups, 1):
    print(f"    {i:2d}. {sg}")

# -----------------------------
# Update resolve_subgroups to use the unified tax_node_name list
# -----------------------------
def resolve_subgroups(requested: list) -> list:
    """Resolve requested subgroup names to what's actually in tax_node_name (unified)."""
    out = []
    missing = []
    
    for r in requested:
        if r in available_subgroups:
            out.append(r)
            continue
        # contains match
        contains = [a for a in available_subgroups if r.lower() in a.lower()]
        if len(contains) == 1:
            out.append(contains[0])
            continue
        # fuzzy
        sugg = difflib.get_close_matches(r, available_subgroups, n=6, cutoff=0.55)
        missing.append((r, sugg))
    
    if missing:
        print("\n⚠️ Missing subgroup names (suggestions):")
        for r, sugg in missing:
            print(f"  - '{r}' -> {sugg}")
    
    return out

# Test resolve with some examples
print("\n✓ Resolve check examples:")
print("  Explicit:", resolve_subgroups(["Explicit Sexual Acts"]))
print("  Domestic:", resolve_subgroups(["Domestic Spaces & Routines"]))
print("  Miscomm:", resolve_subgroups(["Communication & Miscommunication"]))
print("  Attraction:", resolve_subgroups(["Attraction & Sexual Tension"]))
print("  Commitment:", resolve_subgroups(["Reconciliation, Commitments & HEA", "Relationship Stage & Commitment"]))



✓ Available subgroup nodes for CORE matching: 28
  Full list:
     1. Ambivalence & Internal Conflict
     2. Attraction & Sexual Tension
     3. Beliefs, Values & Moral Reflection
     4. Body Parts & Physical Reactions
     5. Bonding, Everyday Intimacy & Growth
     6. Community, Norms & Social Events
     7. Conflict, Distance & Breakup Threats
     8. Domestic Spaces & Routines
     9. Exercise & Physical Activity
    10. Explicit Sexual Acts
    11. Family & Kinship
    12. Friends & Social Circles
    13. Hero's Elite Work & Business World
    14. Heroine's Work & Professional Identity
    15. Interpersonal Non-Romantic Conflict
    16. Kissing & Non-Explicit Affection
    17. Law, Medicine, Education & Formal Institutions
    18. Meeting, First Impressions & Setup
    19. Money, Housing & Economic Security
    20. Negative Emotions & Distress
    21. Noise / Technical / Paratext
    22. Positive Emotions & Security
    23. Public & Leisure Spaces
    24. Reconciliation, Commit

## 7.8. Update CORE Dictionary with Correct Subgroup Names

Update the CORE dictionary to use subgroup names that actually exist in `tax_sub_name`. Check the output above to see the exact names.


In [17]:
# Update CORE dictionary - use resolve_subgroups to match actual tax_sub_name values
# This will print warnings for any mismatches

CORE_RESOLVED = {}
for comp_name, requested_subgroups in CORE.items():
    resolved = resolve_subgroups(requested_subgroups)
    CORE_RESOLVED[comp_name] = resolved
    if len(resolved) == 0:
        print(f"⚠️  WARNING: {comp_name} has NO matching subgroups! Check the requested names.")

print("\n✓ CORE dictionary resolved to actual tax_sub_name values")
print(f"  Composites: {len(CORE_RESOLVED)}")
for comp, subs in CORE_RESOLVED.items():
    if subs:
        print(f"    {comp}: {len(subs)} subgroups")
    else:
        print(f"    {comp}: ⚠️  NO SUBGROUPS (will fail)")



⚠️ Missing subgroup names (suggestions):
  - 'Relationship Stage & Commitment' -> ['Reconciliation, Commitments & HEA']
  - 'Rupture, Separation & Reconciliation' -> []
⚠️  WARNING: A_reassurance_commitment has NO matching subgroups! Check the requested names.

⚠️ Missing subgroup names (suggestions):
  - 'Soft Affection & Non-Sexual Touch' -> ['Attraction & Sexual Tension']
  - 'Sexual Arousal & Foreplay' -> []
⚠️  WARNING: B_mutual_intimacy has NO matching subgroups! Check the requested names.

⚠️ Missing subgroup names (suggestions):
  - 'Money, Wealth & Economic Security' -> ['Money, Housing & Economic Security']
  - 'Luxury Lifestyle & Status Performance' -> []
  - 'Work & Professional Life' -> ["Heroine's Work & Professional Identity", 'Shared Workplaces & Professional Interaction']
⚠️  WARNING: D_luxury_status has NO matching subgroups! Check the requested names.

⚠️ Missing subgroup names (suggestions):
  - 'Physical Threats & Violence' -> ['Violence, Threats & Coercion']
  - 

## 8.1. Quick Test: Does Matching Work?

Test a simple mask to verify the rule engine is working.


## 10. Taxonomy-First Composite Building & Book-Level Aggregation

Build composites using taxonomy-first approach and aggregate to book level with hypothesis indices.


In [18]:
# This cell implements the complete taxonomy-first approach,
# using improved mapping for missing subgroup names.

# Prepare standardized taxonomy columns
if "taxonomy_main_name" not in topic_spine.columns:
    for cand in [
        "taxonomy_main_name_taxmap",
        "taxonomy_main_name_lookup",
        "taxonomy_main_name_x",
        "taxonomy_main_name_y",
    ]:
        if cand in topic_spine.columns:
            topic_spine["taxonomy_main_name"] = topic_spine[cand]
            break

if "taxonomy_main_group" not in topic_spine.columns:
    for cand in [
        "taxonomy_main_group_taxmap",
        "taxonomy_main_group_lookup",
        "taxonomy_main_group_x",
        "taxonomy_main_group_y",
    ]:
        if cand in topic_spine.columns:
            topic_spine["taxonomy_main_group"] = topic_spine[cand]
            break

# Define mapping for missing subgroup names based on suggestions
missing_subgroup_map = {
    # 'Old Name': 'Suggested Replacement',
    "Relationship Stage & Commitment": "Reconciliation, Commitments & HEA",
    "Soft Affection & Non-Sexual Touch": "Attraction & Sexual Tension",
    "Money, Wealth & Economic Security": "Money, Housing & Economic Security",
    "Work & Professional Life": "Heroine's Work & Professional Identity",  # Prefer one, you can adjust
    "Physical Threats & Violence": "Violence, Threats & Coercion",
    "Guilt, Shame & Moral Conflict": "Beliefs, Values & Moral Reflection",
    "Inner Conflict, Decisions & Reflection": "Beliefs, Values & Moral Reflection",
    "Domestic Spaces & Home Life": "Domestic Spaces & Routines",
    "Positive Emotions & Safety": "Positive Emotions & Security",
    "Sensory Impressions": "Meeting, First Impressions & Setup",
    "Interpersonal Conflict & Betrayal": "Interpersonal Non-Romantic Conflict",
    # Add further replacements as appropriate, based on feedback
}

def resolve_subgroup_names(subgroup_names):
    """
    Replace missing or deprecated subgroup names with best suggestions.
    subgroup_names: list of str or set of str
    Returns: list of standardized subgroup names
    """
    if isinstance(subgroup_names, str):
        subgroup_names = [subgroup_names]
    result = []
    for name in subgroup_names:
        std = missing_subgroup_map.get(name, name)
        result.append(std)
    return list(sorted(set(result)))

available_subgroups = sorted(
    set(
        resolve_subgroup_names(
            [x for x in topic_spine["taxonomy_main_name"].dropna().astype(str).unique().tolist()]
        )
    )
)

def sg(x):
    # Accept a list/set, replace names if needed
    if isinstance(x, (list, set, tuple)):
        return resolve_subgroup_names(x)
    if isinstance(x, str):
        return resolve_subgroup_names([x])
    return []

CORE_RESOLVED = {k: sg(v) for k, v in CORE.items()}

# ---- Composite mask definitions (identical logic) ----

REFINE = {
    "A_reassurance_commitment": {
        "include_any": [
            {"secondary_dim_any": {"activity": COMMITMENT_ACTS}},
            {"secondary_dim_any": {"setting": COMMITMENT_SETTINGS}},
            {"radway_main_any": ["commit", "love declaration", "identity restored"]},
        ],
        "exclude_any": [
            {"secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM}},
            {"secondary_dim_any": {"activity": EXPLICIT_ACTIVITIES}},
        ]
    },
    "B_mutual_intimacy": {
        "include_any": [
            {"secondary_dim_any": {"activity": SOFT_AFFECTION_ACTIVITIES}},
            {"primary_any": ["physical_affection", "sexual_tension", "romance_core"]},
        ],
        "exclude_any": [
            {"secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM}},
            {"secondary_dim_any": {"activity": EXPLICIT_ACTIVITIES}},
            {"keyword_any": EXPLICIT_KEYWORDS},
        ]
    },
    "C_explicit": {
        "include_any": [
            {"secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM}},
            {"secondary_dim_any": {"activity": EXPLICIT_ACTIVITIES}},
            {"keyword_any": EXPLICIT_KEYWORDS},
            {"primary_any": ["sexual_content"]},
        ],
        "exclude_any": [
            {"secondary_dim_any": {"activity": ["kissing", "making_out"]}},
        ]
    },
    "D_luxury_status": {
        "include_any": [{"primary_any": ["business_setting", "business_or_work"]}],
        "include_all": [
            {
                "secondary_dim_any": {"setting": [
                    "boardroom", "press_conference", "hotel_suite", "airplane",
                    "boat", "cockpit", "pool", "club", "gallery", "home_office"
                ]},
                "keyword_any": [
                    "penthouse", "private jet", "designer", "chauffeur", "paparazzi", "yacht", "limousine"
                ],
            },
            {
                "secondary_dim_any": {"activity": ["shopping", "wine_tasting", "partying"]},
                "keyword_any": [
                    "designer", "penthouse", "private jet", "paparazzi", "chauffeur", "yacht"
                ],
            },
        ],
        "exclude_any": [{"primary_any": ["work_or_school"]}],
    },
    "E_coercion_brutality_danger": {
        "include_any": [
            {"secondary_dim_any": {"activity": [
                "planning_murder", "threatening", "calling_police","threat_assessment"
            ]}},
            {"secondary_dim_any": {"setting": [
                "warehouse", "desolate", "police_station"
            ]}},
            {"keyword_any": [
                "gun", "knife", "kidnap", "hostage", "torture", "coerc", "violence"
            ]},
        ],
        "exclude_any": [{"secondary_dim_any": {"activity": [
            "discussion", "discussing", "conversation"
        ]}}],
    },
    "F_angst_negative_affect": {
        "include_any": [
            {"primary_any": ["emotional_content", "relationship_conflict"]},
            {"secondary_dim_any": {"activity": [
                "crying", "worrying", "struggling", "whining", "scowling", "frowning"
            ]}},
            {"secondary_dim_any": {"activity": [
                "arguing", "argument", "rejecting", "lying", "silent_staring"
            ]}},
        ],
        "exclude_any": [{"secondary_dim_any": {"activity": [
            "planning_murder", "calling_police"
        ]}}],
    },
    "G_courtship_rituals_gifts": {
        "include_any": [{"secondary_dim_any": {"activity": RITUAL_MARKERS}}],
        "include_all": [
            {"secondary_dim_any": {"activity": RITUAL_MARKERS}},
            {"secondary_dim_any": {"setting": RITUAL_SETTINGS}, "primary_any": ["social_setting"]},
        ],
        "exclude_any": [
            {"secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM}},
            {"secondary_dim_any": {"activity": EXPLICIT_ACTIVITIES}},
        ],
    },
    "H_domestic_nesting": {
        "include_any": [
            {"primary_any": ["domestic_life"]},
            {"secondary_dim_any": {"setting": [
                "home", "apartment", "living_room", "kitchen", "bathroom"
            ]}},
        ],
        "exclude_any": [
            {"secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM}},
            {"secondary_dim_any": {"activity": EXPLICIT_ACTIVITIES}},
        ]
    },
    "I_humor_lightness": {
        "include_any": [{"secondary_dim_any": {"activity": [
            "laughing", "sarcasm", "teasing", "horseplay"
        ]}}],
        "exclude_any": [{"primary_any": ["nonfiction_or_technical"]}],
    },
    "J_social_support_kin": {
        "include_any": [
            {"secondary_dim_any": {"relationship": ["family"]}},
            {"secondary_dim_any": {"activity": [
                "parenting", "visiting", "celebrating", "gathering"
            ]}},
            {"primary_any": ["social_setting"]},
        ]
    },
    "K_professional_intrusion": {
        "include_any": [
            {"primary_any": [
                "business_or_work", "business_setting", "work_or_school",
            ]},
            {"secondary_dim_any": {"setting": [
                "office", "boardroom", "school", "seminar", "press_conference", "home_office"
            ]}},
            {"secondary_dim_any": {"activity": [
                "working", "discussing_business", "negotiating"
            ]}},
        ]
    },
    "L_vices_addictions": {
        "include_any": [
            {"secondary_dim_any": {"activity": VICE_ACTS}},
            {"secondary_dim_any": {"setting": VICE_SETTINGS}},
            {"keyword_any": ["cigarette", "drug", "vodka", "whiskey", "rehab"]},
        ],
        "exclude_any": [{"secondary_dim_any": {"activity": ["dinner", "invitation"]}}],
    },
    "M_health_recovery_growth": {
        "include_any": [
            {"secondary_dim_any": {"activity": HEALTH_ACTS}},
            {"secondary_dim_any": {"setting": HEALTH_SETTINGS}},
            {"keyword_any": [
                "hospital", "doctor", "nurse", "therapy", "injury", "healing", "recovery"
            ]},
        ]
    },
    "N_separation_reunion": {
        "include_any": [
            {"secondary_dim_any": {"activity": [
                "reunion", "goodbyes", "visiting"
            ]}},
            {"keyword_any": [
                "breakup", "goodbye", "left", "missed", "distance", "separation", "reunion"
            ]},
        ]
    },
    "O_aesthetics_appearance": {
        "include_any": [
            {"secondary_dim_any": {"activity": APPEARANCE_ACTS}},
            {"keyword_any": [
                "dress", "heels", "lipstick", "perfume", "suit", "tie",
                "lingerie", "makeup", "hair"
            ]},
        ],
        "exclude_any": [{"secondary_dim_any": {"sexual": ["nudity"]}}],
    },
    "P_tech_media_presence": {
        "include_any": [
            {"secondary_dim_any": {"setting": TECH_SETTINGS}},
            {"secondary_dim_any": {"activity": TECH_ACTS}},
        ]
    },
    "Q_miscommunication": {
        "include_any": [{"secondary_dim_any": {"activity": MISCOMM_STRONG}}],
        "exclude_any": [{"secondary_dim_any": {"activity": ["planning_murder", "calling_police"]}}],
    },
    "Q_repair": {
        "include_any": [{"secondary_dim_any": {"activity": REPAIR_ACTS}}],
        "exclude_any": [
            {"secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM}},
            {"secondary_dim_any": {"activity": EXPLICIT_ACTIVITIES}},
        ],
    },
    "R_protectiveness": {
        "include_any": [
            {"secondary_dim_any": {"activity": ["intervening", "warning", "offering", "preparing", "threat_assessment"] + HEALTH_ACTS}},
            {"keyword_any": ["protect", "safe", "shield", "guard", "security", "keep you safe"]},
        ]
    },
    "R_jealousy": {
        "include_any": [
            {"secondary_dim_any": {"activity": ["checking_phone", "questioning", "arguing", "threatening"]}},
            {"keyword_any": ["jealous", "possess", "mine", "don't talk to", "belongs to me"]},
        ]
    },
}

FALLBACK = {
    "C_explicit": {"include_any": {"secondary_dim_any": {"sexual": EXPLICIT_SEXUAL_DIM}}},
    "P_tech_media_presence": {"include_any": {"secondary_dim_any": {"setting": TECH_SETTINGS, "activity": TECH_ACTS}}},
}

masks_tf = {}
audit_rows = []
FORCE_TAXONOMY_SHARE = 0.70

# Patch: Build a DataFrame with all columns needed for build_mask_logic as string columns
def get_col_safe(df, col):
    if col in df.columns:
        return df[col]
    else:
        # Create dummy empty strings column of the appropriate length and Series type
        return pd.Series([""] * len(df), index=df.index)

topic_spine_for_mask = topic_spine.copy()
for col in ["label", "scene_summary", "keywords"]:
    # Patch these three columns with empty string if absent or if the dtype is object (but some cells missing)
    if col not in topic_spine_for_mask.columns:
        topic_spine_for_mask[col] = ""
    else:
        # forcibly cast (with fillna) to string to avoid accidental object or float errors
        topic_spine_for_mask[col] = topic_spine_for_mask[col].fillna("").astype(str)

for comp_name, core_subgroups in CORE_RESOLVED.items():
    refine = REFINE.get(comp_name, None)
    fallback = FALLBACK.get(comp_name, None)
    mask, aud = taxonomy_first_mask(
        topic_spine_for_mask,
        core_subgroups=core_subgroups if core_subgroups else [],
        refine_spec=refine,
        fallback_spec=fallback,
        force_taxonomy_share=FORCE_TAXONOMY_SHARE,
        mass_col="prevalence" if "prevalence" in topic_spine_for_mask.columns else "mass",
        weight_col="w_eff",
        name=comp_name
    )
    mask = mask & (topic_spine_for_mask["w_eff"] > 0)
    masks_tf[comp_name] = mask
    audit_rows.append(aud)

audit = pd.DataFrame(audit_rows).sort_values("share_core_mass")
audit_path_csv = OUTPUT_DIR / "composite_taxonomy_first_audit.csv"
audit.to_csv(audit_path_csv, index=False)

print("\n✓ Taxonomy-first composite masks built:")
for k, m in masks_tf.items():
    print(f"  {k}: {int(m.sum())} topics")
print(f"\n✓ Saved audit (CSV): {audit_path_csv}")
display(audit)


⚠️  [C_explicit] core mass share 0.70 < 0.70. Tighten fallback/refine rules.
⚠️  [P_tech_media_presence] core mass share 0.00 < 0.70. Tighten fallback/refine rules.

✓ Taxonomy-first composite masks built:
  A_reassurance_commitment: 0 topics
  B_mutual_intimacy: 7 topics
  C_explicit: 14 topics
  D_luxury_status: 0 topics
  E_coercion_brutality_danger: 2 topics
  F_angst_negative_affect: 3 topics
  G_courtship_rituals_gifts: 0 topics
  H_domestic_nesting: 17 topics
  I_humor_lightness: 0 topics
  J_social_support_kin: 1 topics
  K_professional_intrusion: 1 topics
  L_vices_addictions: 0 topics
  M_health_recovery_growth: 0 topics
  N_separation_reunion: 0 topics
  O_aesthetics_appearance: 0 topics
  P_tech_media_presence: 3 topics
  Q_miscommunication: 0 topics
  Q_repair: 1 topics
  R_protectiveness: 1 topics
  R_jealousy: 0 topics

✓ Saved audit (CSV): /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/in

,composite,n_topics_final,n_topics_core_in_final,n_topics_fallback_only,total_mass,core_mass,share_core_mass
0,A_reassurance_commitment,0,0,0,0.000000,0.000000,0.000000
16,Q_miscommunication,0,0,0,0.000000,0.000000,0.000000
15,P_tech_media_presence,4,0,4,0.008021,0.000000,0.000000
14,O_aesthetics_appearance,0,0,0,0.000000,0.000000,0.000000
13,N_separation_reunion,0,0,0,0.000000,0.000000,0.000000
12,M_health_recovery_growth,0,0,0,0.000000,0.000000,0.000000
11,L_vices_addictions,0,0,0,0.000000,0.000000,0.000000
8,I_humor_lightness,0,0,0,0.000000,0.000000,0.000000
19,R_jealousy,0,0,0,0.000000,0.000000,0.000000
6,G_courtship_rituals_gifts,0,0,0,0.000000,0.000000,0.000000


In [19]:
# -----------------------------
# Aggregate composites per book (book_topic_probs)
# -----------------------------
book_topic_probs = pd.read_parquet(BOOK_TOPIC_PROBS_PATH)
book_topic_probs["topic_id"] = book_topic_probs["topic_id"].astype(int)

btp = book_topic_probs.merge(topic_spine[["topic_id","w_eff"]], on="topic_id", how="left")
btp["w_eff"] = btp["w_eff"].fillna(0.0)

# Add composite columns
for comp, mask in masks_tf.items():
    comp_ids = set(topic_spine.loc[mask, "topic_id"].tolist())
    btp[comp] = np.where(btp["topic_id"].isin(comp_ids), btp["prob"] * btp["w_eff"], 0.0)

comp_cols = list(masks_tf.keys())
book_composites = btp.groupby("book_id", as_index=False)[comp_cols].sum()

# -----------------------------
# Hypothesis indices
# -----------------------------
eps = 1e-9

# Love depth = A + B
if "A_reassurance_commitment" in book_composites.columns and "B_mutual_intimacy" in book_composites.columns:
    book_composites["LoveDepth"] = book_composites["A_reassurance_commitment"] + book_composites["B_mutual_intimacy"]
else:
    book_composites["LoveDepth"] = 0.0

# H1 Love-over-Sex
if "C_explicit" in book_composites.columns:
    book_composites["H1_balance_love_over_sex"] = book_composites["LoveDepth"] - book_composites["C_explicit"]
    book_composites["H1_logratio_love_over_sex"] = np.log((book_composites["LoveDepth"] + eps) / (book_composites["C_explicit"] + eps))
else:
    book_composites["H1_balance_love_over_sex"] = np.nan
    book_composites["H1_logratio_love_over_sex"] = np.nan

# H2 HEA index = A + G
if "G_courtship_rituals_gifts" in book_composites.columns:
    book_composites["H2_HEA_index"] = book_composites.get("A_reassurance_commitment", 0.0) + book_composites["G_courtship_rituals_gifts"]
else:
    book_composites["H2_HEA_index"] = np.nan

# H3 Luxury × Love
if "D_luxury_status" in book_composites.columns:
    book_composites["H3_luxury_x_love"] = book_composites["D_luxury_status"] * book_composites["LoveDepth"]
else:
    book_composites["H3_luxury_x_love"] = np.nan

# H4 Protectiveness vs Jealousy
if "R_protectiveness" in book_composites.columns and "R_jealousy" in book_composites.columns:
    book_composites["H4_protect_minus_jealous"] = book_composites["R_protectiveness"] - book_composites["R_jealousy"]
else:
    book_composites["H4_protect_minus_jealous"] = np.nan

# H5 Darkness vs Tenderness: (E + F) - B
if "E_coercion_brutality_danger" in book_composites.columns and "F_angst_negative_affect" in book_composites.columns:
    dark = book_composites["E_coercion_brutality_danger"] + book_composites["F_angst_negative_affect"]
    tender = book_composites.get("B_mutual_intimacy", 0.0)
    book_composites["H5_dark_vs_tender"] = dark - tender
    book_composites["H5_logratio_dark_vs_tender"] = np.log((dark + eps) / (tender + eps))
else:
    book_composites["H5_dark_vs_tender"] = np.nan
    book_composites["H5_logratio_dark_vs_tender"] = np.nan

# Q balance: repair vs miscomm
if "Q_repair" in book_composites.columns and "Q_miscommunication" in book_composites.columns:
    book_composites["Q_balance_repair_minus_miscomm"] = book_composites["Q_repair"] - book_composites["Q_miscommunication"]
    book_composites["Q_logratio_repair_vs_miscomm"] = np.log((book_composites["Q_repair"] + eps) / (book_composites["Q_miscommunication"] + eps))
else:
    book_composites["Q_balance_repair_minus_miscomm"] = np.nan
    book_composites["Q_logratio_repair_vs_miscomm"] = np.nan

# -----------------------------
# Save outputs (CSV + parquet)
# -----------------------------
# Topic spine
topic_spine_out_csv = OUTPUT_DIR / "topic_spine_for_indexing.csv"
topic_spine_out_parq = OUTPUT_DIR / "topic_spine_for_indexing.parquet"
topic_spine.to_csv(topic_spine_out_csv, index=False)
topic_spine.to_parquet(topic_spine_out_parq, index=False)

# Topic to composite membership
mask_rows = []
for comp, mask in masks_tf.items():
    comp_ids = topic_spine.loc[mask, "topic_id"].astype(int).tolist()
    for tid in comp_ids:
        mask_rows.append({"topic_id": tid, "composite": comp})
mask_df = pd.DataFrame(mask_rows)
mask_out_csv = OUTPUT_DIR / "topic_to_composite_membership.csv"
mask_df.to_csv(mask_out_csv, index=False)

# Book composites and hypothesis indices
book_out_csv = OUTPUT_DIR / "book_composites_and_hypothesis_indices.csv"
book_out_parq = OUTPUT_DIR / "book_composites_and_hypothesis_indices.parquet"
book_composites.to_csv(book_out_csv, index=False)
book_composites.to_parquet(book_out_parq, index=False)

print("\n✓ Saved outputs:")
print("  -", topic_spine_out_csv)
print("  -", topic_spine_out_parq)
print("  -", mask_out_csv)
print("  -", audit_path_csv)
print("  -", book_out_csv)
print("  -", book_out_parq)

display(book_composites.head(10))



✓ Saved outputs:
  - /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/indexing_hypothesis_testing/topic_spine_for_indexing.csv
  - /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/indexing_hypothesis_testing/topic_spine_for_indexing.parquet
  - /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/indexing_hypothesis_testing/topic_to_composite_membership.csv
  - /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/indexing_hypothesis_testing/composite_taxonomy_first_audit.csv
  - /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/indexing_hypothesis_testing/book_composites_and_hypothesis_indices.cs

,book_id,A_reassurance_commitment,B_mutual_intimacy,C_explicit,D_luxury_status,E_coercion_brutality_danger,F_angst_negative_affect,G_courtship_rituals_gifts,H_domestic_nesting,I_humor_lightness,...,LoveDepth,H1_balance_love_over_sex,H1_logratio_love_over_sex,H2_HEA_index,H3_luxury_x_love,H4_protect_minus_jealous,H5_dark_vs_tender,H5_logratio_dark_vs_tender,Q_balance_repair_minus_miscomm,Q_logratio_repair_vs_miscomm
0,104659050,0.0,0.019670,0.055717,0.0,0.003948,0.005796,0.0,0.031164,0.0,...,0.019670,-0.036048,-1.041220,0.0,0.0,0.000502,-0.009926,-0.702462,0.001818,14.413371
1,11266880,0.0,0.017679,0.039314,0.0,0.004027,0.007761,0.0,0.030973,0.0,...,0.017679,-0.021635,-0.799221,0.0,0.0,0.000705,-0.005890,-0.405252,0.000909,13.719825
2,123257687,0.0,0.013648,0.033437,0.0,0.004754,0.008245,0.0,0.030577,0.0,...,0.013648,-0.019789,-0.896102,0.0,0.0,0.000688,-0.000648,-0.048652,0.000936,13.748887
3,123446478,0.0,0.017052,0.043791,0.0,0.003542,0.007796,0.0,0.029024,0.0,...,0.017052,-0.026739,-0.943151,0.0,0.0,0.000584,-0.005714,-0.408100,0.001634,14.306828
4,127305713,0.0,0.013301,0.034942,0.0,0.005242,0.007130,0.0,0.032051,0.0,...,0.013301,-0.021641,-0.965866,0.0,0.0,0.000624,-0.000928,-0.072344,0.001136,13.943447
5,149105520,0.0,0.025142,0.039563,0.0,0.004156,0.006724,0.0,0.032194,0.0,...,0.025142,-0.014421,-0.453359,0.0,0.0,0.000534,-0.014263,-0.837665,0.000855,13.658501
6,15197,0.0,0.018099,0.049469,0.0,0.003375,0.006903,0.0,0.037689,0.0,...,0.018099,-0.031370,-1.005482,0.0,0.0,0.000804,-0.007821,-0.565830,0.001111,13.920817
7,161913,0.0,0.012081,0.036106,0.0,0.003154,0.006384,0.0,0.024305,0.0,...,0.012081,-0.024024,-1.094783,0.0,0.0,0.000599,-0.002544,-0.236394,0.001370,14.130327
8,17561022,0.0,0.014909,0.032906,0.0,0.004409,0.006561,0.0,0.035288,0.0,...,0.014909,-0.017997,-0.791716,0.0,0.0,0.001013,-0.003938,-0.306750,0.001631,14.304799
9,1756703,0.0,0.014055,0.036198,0.0,0.011126,0.006708,0.0,0.028018,0.0,...,0.014055,-0.022143,-0.946046,0.0,0.0,0.000497,0.003779,0.238148,0.000787,13.576192


In [20]:
# One super-fast "does matching work at all?" test
test_spec = {"include_any": {"primary_any": ["sexual_content"]}}
test_mask = build_mask_logic(topic_spine, test_spec)
print("sexual_content topics:", int(test_mask.sum()))
display(topic_spine.loc[test_mask, ["topic_id","primary_categories","secondary_categories"]].head(10))


sexual_content topics: 27


,topic_id,primary_categories,secondary_categories
1,1,"physical_affection, sexual_content","setting:bedroom, activity:kissing"
2,2,"sexual_content, romance_core","setting:bedroom, activity:oral_sex, sexual:cli..."
15,15,"sexual_content, romance_core","setting:elevator, activity:making_out, sexual:..."
19,19,"sexual_content, romance_core","setting:bedroom, activity:foreplay, sexual:tou..."
42,42,"romance_core, sexual_content","setting:bedroom, activity:exploring_virginity"
47,47,"physical_affection, sexual_content","setting:bedroom, activity:kissing"
55,55,"sexual_content, domestic_life","setting:bedroom, activity:BDSM, sexual:dominatrix"
102,102,"sexual_content, romance_core","setting:bedroom, activity:kissing, sexual:hair..."
103,103,"sexual_content, domestic_life","setting:bedroom, sexual:nudity"
129,129,"sexual_content, domestic_life","setting:bedroom, activity:checking_condom"


## 8. Build Composite Masks

Build boolean masks for each composite using the extended rule engine.


In [21]:
# Fix: Defensive ensure series for keyword columns to avoid .fillna on str error
masks = {}
for name, spec in COMPOSITES.items():
    try:
        # Defensive: ensure all fields needed by _rule_to_mask as series, not accidentally str
        expected_text_fields = ["label", "scene_summary", "keywords"]
        for fld in expected_text_fields:
            if fld in topic_spine and isinstance(topic_spine[fld], pd.Series):
                continue
            elif fld in topic_spine and isinstance(topic_spine[fld], str):
                # Replace single string (unexpected) with all-blank series
                topic_spine[fld] = pd.Series([""] * len(topic_spine), index=topic_spine.index)
            elif fld not in topic_spine:
                # If missing, inject blank series
                topic_spine[fld] = pd.Series([""] * len(topic_spine), index=topic_spine.index)
        masks[name] = build_mask_logic(topic_spine, spec)
    except AttributeError as e:
        print(f"Error building mask for composite '{name}': {e}")
        print("  This often means that a column expected to be a DataFrame is actually a string.")
        print("  Please check if your DataFrame is structured as expected and all referenced columns exist.")
        raise

# Optional: enforce a hard exclusion gate (weight > 0)
try:
    w_mask = (topic_spine["w_eff"] > 0)
except Exception as e:
    print(f"Error computing weight filter mask: {e}")
    raise

masks = {k: (v & w_mask) for k, v in masks.items()}

print("✓ Composite masks built")
for name, mask in masks.items():
    print(f"  {name}: {int(mask.sum())} topics")


✓ Composite masks built
  A_commitment_hea: 150 topics
  B_mutual_intimacy: 166 topics
  C_explicit: 19 topics
  D_luxury_status: 17 topics
  E_threat_danger: 82 topics
  F_angst_negative: 118 topics
  G_rituals_gifts: 227 topics
  H_domestic_nesting: 132 topics
  I_humor_lightness: 7 topics
  J_social_support: 40 topics
  K_professional_frame: 20 topics
  L_vices: 3 topics
  M_health_recovery: 3 topics
  O_appearance: 9 topics
  Q_miscommunication: 121 topics
  Q_repair: 16 topics
  R_protectiveness: 5 topics
  R_jealousy: 30 topics
  S_scene_anchors: 159 topics


## 9. Sanity Checks

Check mask counts and verify B vs C separation (should have minimal overlap).


In [22]:
mask_audit = pd.DataFrame({
    "composite": list(masks.keys()),
    "n_topics": [int(m.sum()) for m in masks.values()],
}).sort_values("n_topics", ascending=False)

display(mask_audit)

# Check B vs C separation
if "B_mutual_intimacy" in masks and "C_explicit" in masks:
    both = masks["B_mutual_intimacy"] & masks["C_explicit"]
    print("B ∩ C overlap topics:", int(both.sum()))
    if both.sum() > 0:
        display(topic_spine.loc[both, ["topic_id","primary_categories","secondary_categories","label","keywords"]].head(25))
    else:
        print("✓ No overlap between B and C (good separation)")
else:
    print("⚠ B_mutual_intimacy or C_explicit not found in masks")

,composite,n_topics
6,G_rituals_gifts,227
1,B_mutual_intimacy,166
18,S_scene_anchors,159
0,A_commitment_hea,150
7,H_domestic_nesting,132
14,Q_miscommunication,121
5,F_angst_negative,118
4,E_threat_danger,82
9,J_social_support,40
17,R_jealousy,30


B ∩ C overlap topics: 6


,topic_id,primary_categories,secondary_categories,label,keywords
42,42,"romance_core, sexual_content","setting:bedroom, activity:exploring_virginity",,
162,162,"sexual_content, romance_core","setting:bedroom, activity:teasing",,
207,207,"sexual_content, romance_core","setting:bedroom, activity:scent_smelling",,
208,208,"romance_core, sexual_content","setting:harem, activity:offering",,
274,274,"social_setting, sexual_content","setting:club, activity:partying",,
338,338,"romance_core, sexual_content","setting:dorm_room, activity:sexual_awakening",,
